In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets \
    sentence-transformers faiss-gpu rouge-score nltk bert-score \
    matplotlib seaborn scipy scikit-learn

import os, pickle, numpy as np, re, torch
from collections import Counter
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import pearsonr

BASE = 'path/to/your/directory'
os.makedirs(f'{BASE}/results/p3', exist_ok=True)
os.makedirs(f'{BASE}/plots', exist_ok=True)

In [ ]:
from google.colab import drive
import os

# Create a new mount point directory
new_mount_point = '/content/gdrive'
os.makedirs(new_mount_point, exist_ok=True)

drive.mount(new_mount_point, force_remount=True)

BASE = f'{new_mount_point}/MyDrive/your_folder'  # Update this to your actual folder path
os.makedirs(f'{BASE}/results/p3', exist_ok=True) # Ensure this also reflects the new BASE
os.makedirs(f'{BASE}/plots', exist_ok=True) # Ensure this also reflects the new BASE

with open(f'{BASE}/corpus.pkl','rb') as f: corpus = pickle.load(f)
with open(f'{BASE}/queries.pkl','rb') as f: queries = pickle.load(f)
with open(f'{BASE}/results/p1/vanilla_final.pkl','rb') as f: vanilla = pickle.load(f)
with open(f'{BASE}/results/p1/rag_clean_final.pkl','rb') as f: rag_clean = pickle.load(f)
with open(f'{BASE}/results/p2/all_noisy.pkl','rb') as f: noisy = pickle.load(f)

print(f"✅ Vanilla: {len(vanilla)} | RAG-Clean: {len(rag_clean)} | Noisy configs: {list(noisy.keys())}")

In [ ]:
!pip install -q rouge-score 
import nltk; nltk.download('punkt', quiet=True)
from rouge_score import rouge_scorer as rs
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

rouge  = rs.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method4

def bleu(pred, golds):
    return sentence_bleu([g.lower().split() for g in golds],
                          pred.lower().split(), smoothing_function=smooth)

def rouge_scores(pred, golds):
    r1, rL = 0, 0
    for g in golds:
        s = rouge.score(g, pred)
        r1 = max(r1, s['rouge1'].fmeasure); rL = max(rL, s['rougeL'].fmeasure)
    return r1, rL

def add_trad_metrics(lst):
    for r in lst:
        r['bleu'] = bleu(r['answer'], r['answers'])
        r['rouge1'], r['rougeL'] = rouge_scores(r['answer'], r['answers'])
    return lst

print("Computing traditional metrics...")
vanilla   = add_trad_metrics(vanilla)
rag_clean = add_trad_metrics(rag_clean)
for k in noisy: noisy[k] = add_trad_metrics(noisy[k])
print("✅ BLEU + ROUGE done")

In [ ]:
!pip install -q faiss-cpu # Install faiss-cpu instead of faiss-gpu
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer('all-MiniLM-L6-v2')
index    = faiss.read_index(f'{BASE}/faiss_index.bin')

def retrieve(question, top_k=5):
    q = embedder.encode([question], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q)
    sc, idx = index.search(q, top_k)
    return [{'passage':corpus[i],'score':float(s)} for i,s in zip(idx[0],sc[0])]

In [ ]:
# CWF = α·G + β·U + γ·R
# G = Grounding (answer ↔ retrieved docs semantic sim)
# U = Uncertainty proxy (answer specificity)
# R = Retrieval quality (avg FAISS score)

def grounding(answer, docs, emb):
    if not answer or not docs: return 0.0
    a = emb.encode(answer, convert_to_numpy=True)
    d = emb.encode([x['passage']['text'] for x in docs], convert_to_numpy=True)
    a /= np.linalg.norm(a)+1e-10
    d /= np.linalg.norm(d, axis=1, keepdims=True)+1e-10
    return float(np.max(d @ a))

def uncertainty_proxy(answer):
    """Penalize vague/short/hedging answers (no model logprobs needed)."""
    if not answer: return 0.0
    hedges = ['i don\'t know','not sure','unclear','no information',
              'cannot determine','not in the','no relevant']
    if any(h in answer.lower() for h in hedges): return 0.1
    words = answer.split()
    return min(1.0, len(words)/12.0) if len(words)<12 else 1.0

def retrieval_quality(docs):
    if not docs: return 0.0
    return float(np.mean([d['score'] for d in docs]))

def cwf(answer, docs, emb, alpha=0.52, beta=0.31, gamma=0.17):
    G = grounding(answer, docs, emb)
    U = uncertainty_proxy(answer)
    R = retrieval_quality(docs)
    return {'cwf': alpha*G + beta*U + gamma*R, 'G':G, 'U':U, 'R':R}


In [ ]:
def enrich_with_cwf(results_list, doc_key='retrieved_docs', label=""):
    ckpt = f"{BASE}/checkpoints/cwf_{label}.pkl"
    if os.path.exists(ckpt):
        with open(ckpt,'rb') as f: return pickle.load(f)
    for i, r in enumerate(results_list):
        docs = r.get(doc_key) or retrieve(r['question'])
        r.update(cwf(r['answer'], docs, embedder))
        if (i+1) % 50 == 0:
            with open(ckpt,'wb') as f: pickle.dump(results_list, f)
    with open(ckpt,'wb') as f: pickle.dump(results_list, f)
    print(f"  ✅ CWF done for {label}")
    return results_list

print("Computing CWF scores...")
rag_clean = enrich_with_cwf(rag_clean, 'retrieved_docs', 'rag_clean')
for cfg in noisy:
    noisy[cfg] = enrich_with_cwf(noisy[cfg], 'noisy_docs', cfg)
print("✅ CWF complete")

In [ ]:
# ANDF = w1·Relevance + w2·Consistency + w3·Temporal
# Filters documents scoring below threshold

def andf_score(query, doc_text, all_texts, emb, w1=0.5, w2=0.3, w3=0.2):
    q_n  = emb.encode(query,    convert_to_numpy=True); q_n /= np.linalg.norm(q_n)+1e-10
    d_n  = emb.encode(doc_text, convert_to_numpy=True); d_n /= np.linalg.norm(d_n)+1e-10
    rel  = float(q_n @ d_n)

    if len(all_texts) > 1:
        others = emb.encode([t for t in all_texts if t != doc_text], convert_to_numpy=True)
        others /= np.linalg.norm(others, axis=1, keepdims=True)+1e-10
        con = float(np.mean(others @ d_n))
    else:
        con = 0.8

    yrs = re.findall(r'\b(1[89]\d{2}|20[012]\d)\b', doc_text)
    tmp = float(min(1.0,(np.mean([int(y) for y in yrs])-1970)/(2024-1970))) if yrs else 0.8

    return w1*rel + w2*con + w3*tmp, rel, con, tmp

def andf_filter(query, docs, emb, threshold=0.25):
    if not docs: return docs
    texts  = [d['passage']['text'] for d in docs]
    scored = []
    for d in docs:
        sc, rel, con, tmp = andf_score(query, d['passage']['text'], texts, emb)
        scored.append({**d, 'andf_score':sc, 'andf_rel':rel, 'andf_con':con, 'andf_tmp':tmp})
    kept = [s for s in scored if s['andf_score'] >= threshold]
    return kept if kept else [max(scored, key=lambda x: x['andf_score'])]


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

USE_LLAMA = True
HF_TOKEN  = "your_huggingface_token"  # Needed for private models like LLaMA 3.1
MODEL_NAME = ("meta-llama/Llama-3.1-8B-Instruct" if USE_LLAMA
              else "microsoft/Phi-3-mini-4k-instruct")
token_arg = HF_TOKEN if USE_LLAMA else None

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.float16,
                          bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token_arg)
llm       = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                                  device_map="auto", token=token_arg)
llm.eval()
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

def _fmt(question, docs=None):
    if USE_LLAMA:
        sys  = "You are a helpful assistant. Use the documents to answer accurately."
        ctx  = ("\n\n".join(f"[Doc {i+1}]: {d['passage']['text']}" for i,d in enumerate(docs)) if docs else "")
        user = (f"Documents:\n{ctx}\n\n" if docs else "") + f"Question: {question}\n\nAnswer (1-2 sentences):"
        return (f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{sys}\n"
                f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user}"
                f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n")
    else:
        ctx = ("\n\n".join(f"[Doc {i+1}]: {d['passage']['text']}" for i,d in enumerate(docs)) if docs else "")
        return f"<|user|>\n{'Documents:\n'+ctx+chr(10)+chr(10) if docs else ''}Question: {question}\nAnswer briefly:<|end|>\n<|assistant|>\n"

def generate(prompt, max_new_tokens=80):
    inp = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    with torch.no_grad():
        out = llm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                            temperature=1.0, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def norm(s):
    s = re.sub(r'\b(a|an|the)\b',' ',s.lower()); return ' '.join(re.sub(r'[^\w\s]','',s).split())
def em(pred,golds): return int(any(norm(g)==norm(pred) for g in golds))
def f1(pred,golds):
    pt=norm(pred).split(); best=0
    for g in golds:
        gt=norm(g).split(); c=Counter(pt)&Counter(gt); nc=sum(c.values())
        if nc==0: continue
        pr=nc/len(pt) if pt else 0; rc=nc/len(gt) if gt else 0
        if pr+rc>0: best=max(best,2*pr*rc/(pr+rc))
    return best

In [ ]:
andf_results = {}
TARGET_CONFIGS = [f"{nt}_50" for nt in ['irrelevant','contradictory','outdated','partial','mixed']
                  if f"{nt}_50" in noisy]

for cfg in TARGET_CONFIGS:
    final = f"{BASE}/results/p3/andf_{cfg}_final.pkl"
    ckpt  = f"{BASE}/checkpoints/andf_{cfg}.pkl"

    if os.path.exists(final):
        with open(final,'rb') as f: andf_results[cfg] = pickle.load(f)
        print(f"✅ ANDF {cfg} loaded (F1={np.mean([r['f1'] for r in andf_results[cfg]]):.3f})")
        continue

    results = []
    if os.path.exists(ckpt):
        with open(ckpt,'rb') as f: results = pickle.load(f)
    start = len(results)
    print(f"\n▶ ANDF on {cfg}: resuming from {start}/{len(queries)}")

    for i, q in enumerate(queries[start:], start=start):
        # Pull noisy docs from Person 2 results (if available)
        p2 = noisy[cfg][i] if i < len(noisy.get(cfg,[])) else None
        raw_docs = p2.get('noisy_docs') if p2 else retrieve(q['question'])

        # ANDF filtering
        filtered = andf_filter(q['question'], raw_docs, embedder)
        ans      = generate(_fmt(q['question'], filtered))
        cwf_res  = cwf(ans, filtered, embedder)

        results.append({
            'query_id': q['id'], 'question': q['question'],
            'answers': q['answers'], 'answer': ans,
            'filtered_docs': filtered,
            'n_kept': len(filtered), 'n_original': len(raw_docs),
            'em': em(ans, q['answers']), 'f1': f1(ans, q['answers']),
            **cwf_res
        })

        if (i+1) % 25 == 0 or i == len(queries)-1:
            with open(ckpt,'wb') as f: pickle.dump(results, f)
            print(f"  [{i+1}/{len(queries)}] F1={np.mean([r['f1'] for r in results]):.3f} ✅")

    with open(final,'wb') as f: pickle.dump(results, f)
    andf_results[cfg] = results
    print(f"✅ ANDF {cfg}: F1={np.mean([r['f1'] for r in results]):.3f}")


In [ ]:
def avg(lst, k): return np.mean([r[k] for r in lst if k in r]) if lst else 0

print("\n" + "="*65)
print(f"{'Config':<35} {'EM':>6} {'F1':>6} {'BLEU':>6} {'ROUGE1':>7}")
print("="*65)
print(f"{'Vanilla (no retrieval)':<35} {avg(vanilla,'em'):>6.3f} {avg(vanilla,'f1'):>6.3f} {avg(vanilla,'bleu'):>6.3f} {avg(vanilla,'rouge1'):>7.3f}")
print(f"{'RAG-Clean':<35} {avg(rag_clean,'em'):>6.3f} {avg(rag_clean,'f1'):>6.3f} {avg(rag_clean,'bleu'):>6.3f} {avg(rag_clean,'rouge1'):>7.3f}")
print("-"*65)
for cfg, res in sorted(noisy.items()):
    print(f"{cfg:<35} {avg(res,'em'):>6.3f} {avg(res,'f1'):>6.3f} {avg(res,'bleu'):>6.3f} {avg(res,'rouge1'):>7.3f}")
print("-"*65)
for cfg, res in sorted(andf_results.items()):
    print(f"{'ANDF+'+cfg:<35} {avg(res,'em'):>6.3f} {avg(res,'f1'):>6.3f} {'—':>6} {'—':>7}")
print("="*65)

In [ ]:

import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# ── Part A: CWF by noise level (the real story) ───────
# CWF should DROP as noise increases, proving it tracks
# document grounding — even when BLEU/ROUGE don't drop.

noise_types_a  = ['irrelevant', 'contradictory', 'outdated', 'partial']
noise_levels_a = [25, 50, 75]

print("=== CWF vs BLEU vs ROUGE — Sensitivity to Noise ===")
print(f"{'Config':<28} {'F1':>6} {'BLEU':>7} {'ROUGE1':>8} {'CWF':>7}")
print("─"*60)
clean_cwf  = np.mean([r['cwf']    for r in rag_clean if 'cwf' in r])
clean_bleu = np.mean([r['bleu']   for r in rag_clean])
clean_r1   = np.mean([r['rouge1'] for r in rag_clean])
clean_f1   = np.mean([r['f1']     for r in rag_clean])
print(f"{'RAG-Clean':<28} {clean_f1:>6.3f} {clean_bleu:>7.3f} {clean_r1:>8.3f} {clean_cwf:>7.3f}")
print("─"*60)

cwf_by_config, bleu_by_config, rouge_by_config, f1_by_config = {}, {}, {}, {}
for nt in noise_types_a:
    for lv in noise_levels_a:
        cfg = f"{nt}_{lv}"
        if cfg not in noisy: continue
        res = noisy[cfg]
        cwf_by_config[cfg]   = np.mean([r['cwf']    for r in res if 'cwf' in r]) if any('cwf' in r for r in res) else None
        bleu_by_config[cfg]  = np.mean([r['bleu']   for r in res])
        rouge_by_config[cfg] = np.mean([r['rouge1'] for r in res])
        f1_by_config[cfg]    = np.mean([r['f1']     for r in res])
        cwf_v = cwf_by_config[cfg]
        print(f"{cfg:<28} {f1_by_config[cfg]:>6.3f} {bleu_by_config[cfg]:>7.3f} {rouge_by_config[cfg]:>8.3f} {cwf_v if cwf_v else 0:>7.3f}")

# ── Part B: Divergence metric ─────────────────────────
# Key claim: BLEU/ROUGE STAY HIGH while CWF drops.
# This is the "metric-faithfulness divergence" proof.
print("\n=== Metric-Faithfulness Divergence (50% noise) ===")
print("When noise=50%, how much does each metric DROP from RAG-Clean?")
print(f"{'Noise Type':<20} {'F1 drop':>10} {'BLEU drop':>11} {'ROUGE drop':>12} {'CWF drop':>10}")
print("─"*65)
for nt in noise_types_a:
    cfg = f"{nt}_50"
    if cfg not in noisy: continue
    f1_drop   = clean_f1   - f1_by_config[cfg]
    bleu_drop = clean_bleu - bleu_by_config[cfg]
    rou_drop  = clean_r1   - rouge_by_config[cfg]
    cwf_drop  = (clean_cwf - cwf_by_config[cfg]) if cwf_by_config[cfg] else 0
    print(f"{nt:<20} {f1_drop:>+10.3f} {bleu_drop:>+11.3f} {rou_drop:>+12.3f} {cwf_drop:>+10.3f}")

# ── Part C: Correlation per condition (not pooled) ────
# Pooling clean + noisy confounds correlation.
# Compute within noisy configs only.
noisy_only_cwf, noisy_only_f1 = [], []
noisy_only_bleu, noisy_only_r1 = [], []
for cfg, res in noisy.items():
    for r in res:
        if 'cwf' not in r: continue
        noisy_only_cwf.append(r['cwf'])
        noisy_only_f1.append(r['f1'])
        noisy_only_bleu.append(r['bleu'])
        noisy_only_r1.append(r['rouge1'])

r_cwf_n,  _ = pearsonr(noisy_only_cwf,  noisy_only_f1)
r_bleu_n, _ = pearsonr(noisy_only_bleu, noisy_only_f1)
r_r1_n,   _ = pearsonr(noisy_only_r1,   noisy_only_f1)

print(f"\n=== Within-Noisy Metric ↔ F1 Correlation (n={len(noisy_only_f1)}) ===")
print(f"CWF    r = {r_cwf_n:.3f}")
print(f"BLEU   r = {r_bleu_n:.3f}")
print(f"ROUGE1 r = {r_r1_n:.3f}")

# ── Part D: The real novel finding plot ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CWF: Measuring What BLEU/ROUGE Miss', fontsize=13, fontweight='bold')

# Left: CWF vs BLEU sensitivity to noise (outdated is key)
ax = axes[0]
lv_x = [25, 50, 75]
for nt, color in zip(['irrelevant','contradictory','outdated'],['steelblue','tomato','seagreen']):
    cwf_vals_n  = [cwf_by_config.get(f"{nt}_{l}", 0)  for l in lv_x]
    bleu_vals_n = [bleu_by_config.get(f"{nt}_{l}", 0) for l in lv_x]
    ax.plot(lv_x, cwf_vals_n,  'o-',  color=color, linewidth=2, label=f'{nt} CWF')
    ax.plot(lv_x, bleu_vals_n, 's--', color=color, linewidth=1, alpha=0.5, label=f'{nt} BLEU')

ax.axhline(clean_cwf,  color='black', linestyle='-',  linewidth=1.5, label=f'Clean CWF ({clean_cwf:.3f})')
ax.axhline(clean_bleu, color='gray',  linestyle='--', linewidth=1.5, label=f'Clean BLEU ({clean_bleu:.3f})')
ax.set_xlabel('Noise Level (%)'); ax.set_ylabel('Score')
ax.set_title('CWF (solid) drops faster than BLEU (dashed)\nas noise increases — especially for outdated noise')
ax.legend(fontsize=6.5); ax.set_xticks(lv_x)

# Right: Divergence bar chart — the "metric lies" panel
ax = axes[1]
noise_labels = ['irrelevant', 'contradictory', 'outdated', 'partial']
f1_drops   = [clean_f1   - f1_by_config.get(f"{n}_50", clean_f1)   for n in noise_labels]
bleu_drops = [clean_bleu - bleu_by_config.get(f"{n}_50", clean_bleu) for n in noise_labels]
cwf_drops  = [clean_cwf  - (cwf_by_config.get(f"{n}_50") or clean_cwf) for n in noise_labels]

x = np.arange(len(noise_labels)); w = 0.25
ax.bar(x-w, f1_drops,   w, label='F1 drop',   color='steelblue')
ax.bar(x,   bleu_drops, w, label='BLEU drop',  color='orange')
ax.bar(x+w, cwf_drops,  w, label='CWF drop',   color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(noise_labels)
ax.set_ylabel('Drop from RAG-Clean')
ax.set_title('Metric Divergence at 50% Noise\n(CWF detects faithfulness degradation BLEU misses)')
ax.legend()

plt.tight_layout()
plt.savefig(f'{BASE}/plots/cwf_reframed.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Plot saved: {BASE}/plots/cwf_reframed.png")



In [ ]:

print("\n=== ANDF Recovery — Threshold Sweep ===")
print("Finding the best threshold for each noise type...\n")

# Test thresholds 0.25 → 0.60
thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

def quick_andf_eval(query, docs, threshold):
    """ANDF filter without re-generating — for threshold sweep only."""
    texts  = [d['passage']['text'] for d in docs]
    scored = []
    for d in docs:
        sc, _, _, _ = andf_score(query, d['passage']['text'], texts, embedder)
        scored.append((sc, d))
    kept = [d for sc, d in scored if sc >= threshold]
    return kept if kept else [max(scored, key=lambda x: x[0])[1]]

# Quick threshold sweep on 100 samples (no LLM needed here —
# just check how many docs get filtered per threshold)
print(f"{'Threshold':<12} {'Avg kept/5':>12} {'Irrelevant filtered%':>22}")
print("─"*48)
for thr in thresholds:
    kept_counts, filtered_pct = [], []
    sample = noisy.get('irrelevant_50', [])[:100]
    for r in sample:
        docs   = r.get('noisy_docs', retrieve(r['question']))
        kept   = quick_andf_eval(r['question'], docs, thr)
        n_noisy = round(len(docs) * 0.5)  # expected noisy docs
        kept_counts.append(len(kept))
        filtered_pct.append((len(docs)-len(kept))/len(docs)*100)
    print(f"{thr:<12.2f} {np.mean(kept_counts):>12.2f} {np.mean(filtered_pct):>21.1f}%")

# ── Rerun ANDF with threshold=0.45 ────────────────────
BEST_THRESHOLD = 0.45
print(f"\n▶ Re-running ANDF with threshold={BEST_THRESHOLD}")
print("  (Only re-runs configs where old recovery was < 10%)\n")

andf_results_v2 = {}
for cfg in TARGET_CONFIGS:
    old_res   = andf_results.get(cfg, [])
    old_f1    = np.mean([r['f1'] for r in old_res]) if old_res else 0
    noisy_f1  = np.mean([r['f1'] for r in noisy.get(cfg, [{'f1':0}])])
    old_rec   = (old_f1 - noisy_f1) / (avg(rag_clean,'f1') - noisy_f1 + 1e-10) * 100

    final_v2 = f"{BASE}/results/p3/andf_v2_{cfg}_final.pkl"
    ckpt_v2  = f"{BASE}/checkpoints/andf_v2_{cfg}.pkl"

    if os.path.exists(final_v2):
        with open(final_v2,'rb') as f: andf_results_v2[cfg] = pickle.load(f)
        new_f1 = np.mean([r['f1'] for r in andf_results_v2[cfg]])
        print(f"✅ {cfg} loaded (F1={new_f1:.3f})")
        continue

    results_v2 = []
    if os.path.exists(ckpt_v2):
        with open(ckpt_v2,'rb') as f: results_v2 = pickle.load(f)
    start = len(results_v2)
    print(f"▶ ANDF-v2 ({BEST_THRESHOLD}) on {cfg}: from {start}/500")

    for i, q in enumerate(queries[start:], start=start):
        p2       = noisy[cfg][i] if i < len(noisy.get(cfg,[])) else None
        raw_docs = p2.get('noisy_docs') if p2 else retrieve(q['question'])
        filtered = andf_filter(q['question'], raw_docs, embedder, threshold=BEST_THRESHOLD)
        ans      = generate(_fmt(q['question'], filtered))
        cwf_res  = cwf(ans, filtered, embedder)
        results_v2.append({
            'query_id': q['id'], 'question': q['question'],
            'answers': q['answers'], 'answer': ans,
            'filtered_docs': filtered,
            'n_kept': len(filtered), 'n_original': len(raw_docs),
            'em': em(ans, q['answers']), 'f1': f1(ans, q['answers']),
            **cwf_res
        })
        if (i+1) % 25 == 0 or i == len(queries)-1:
            with open(ckpt_v2,'wb') as f: pickle.dump(results_v2, f)
            print(f"  [{i+1}/500] F1={np.mean([r['f1'] for r in results_v2]):.3f} ✅")

    with open(final_v2,'wb') as f: pickle.dump(results_v2, f)
    andf_results_v2[cfg] = results_v2
    print(f"✅ ANDF-v2 {cfg}: F1={np.mean([r['f1'] for r in results_v2]):.3f}")

# ── Final ANDF Recovery Table ──────────────────────────
print(f"\n=== ANDF Recovery Comparison (threshold {0.25} → {BEST_THRESHOLD}) ===")
print(f"{'Config':<25} {'Noisy F1':>9} {'ANDF v1':>9} {'ANDF v2':>9} {'Rec v1':>9} {'Rec v2':>9}")
print("─"*65)
clean_f1_val = avg(rag_clean, 'f1')
for cfg in sorted(andf_results_v2.keys()):
    nf1  = avg(noisy.get(cfg,[{'f1':0}]), 'f1')
    v1f1 = avg(andf_results.get(cfg,[{'f1':0}]), 'f1')
    v2f1 = avg(andf_results_v2[cfg], 'f1')
    rec1 = (v1f1-nf1)/(clean_f1_val-nf1+1e-10)*100
    rec2 = (v2f1-nf1)/(clean_f1_val-nf1+1e-10)*100
    print(f"{cfg:<25} {nf1:>9.3f} {v1f1:>9.3f} {v2f1:>9.3f} {rec1:>8.1f}% {rec2:>8.1f}%")
avg_rec_v2 = np.mean([(avg(andf_results_v2[c],'f1') - avg(noisy.get(c,[{'f1':0}]),'f1'))
                       /(clean_f1_val - avg(noisy.get(c,[{'f1':0}]),'f1')+1e-10)*100
                       for c in andf_results_v2])
print(f"\n  Average ANDF v2 recovery: {avg_rec_v2:.1f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RAGFail: Core Results', fontsize=14, fontweight='bold')

# Plot 1 — Degradation curve
ax = axes[0,0]
noise_types_plot = ['irrelevant','contradictory','outdated']
levels_pct = [25, 50, 75]
for nt in noise_types_plot:
    f1s = [avg(noisy.get(f"{nt}_{lv}",[{'f1':0}]),'f1') for lv in levels_pct]
    ax.plot(levels_pct, f1s, marker='o', label=nt)
ax.axhline(avg(vanilla,'f1'),    color='black',  linestyle='--', label=f"Vanilla ({avg(vanilla,'f1'):.3f})")
ax.axhline(avg(rag_clean,'f1'), color='green',  linestyle='--', label=f"RAG-Clean ({avg(rag_clean,'f1'):.3f})")
ax.set_xlabel('Noise Level (%)'); ax.set_ylabel('F1')
ax.set_title('Performance Degradation Under Noise')
ax.set_xticks(levels_pct); ax.legend(fontsize=7)

# Plot 2 — Metric divergence
ax = axes[0,1]
cfgs  = ['Vanilla','RAG-Clean'] + [f"{nt}_50" for nt in ['irrelevant','contradictory','outdated']]
data  = {}
for m in ['f1','bleu','rouge1']:
    vals = []
    for c in cfgs:
        r = vanilla if c=='Vanilla' else rag_clean if c=='RAG-Clean' else noisy.get(c,[])
        vals.append(avg(r, m))
    data[m] = vals
x = np.arange(len(cfgs)); w = 0.25
ax.bar(x-w, data['f1'],    w, label='F1 (true)', color='steelblue')
ax.bar(x,   data['bleu'],  w, label='BLEU',      color='orange')
ax.bar(x+w, data['rouge1'],w, label='ROUGE-1',   color='seagreen')
ax.set_xticks(x); ax.set_xticklabels([c.replace('_50','') for c in cfgs], rotation=25, ha='right', fontsize=7)
ax.set_title('Metric Divergence: BLEU/ROUGE vs True F1'); ax.legend(fontsize=7)

# Plot 3 — CWF scatter
ax = axes[1,0]
cwf_vals = noisy_only_cwf
f1_vals = noisy_only_f1
r_cwf = r_cwf_n
sample = min(300, len(cwf_vals))
ax.scatter(cwf_vals[:sample], f1_vals[:sample], alpha=0.3, s=12, color='steelblue')
m, b = np.polyfit(cwf_vals[:sample], f1_vals[:sample], 1)
xl = np.linspace(min(cwf_vals),max(cwf_vals),100)
ax.plot(xl, m*xl+b, 'r-', linewidth=2, label=f'r={r_cwf:.3f}')
ax.set_xlabel('CWF Score'); ax.set_ylabel('F1')
ax.set_title('CWF Correlation with True Faithfulness (F1)'); ax.legend()

# Plot 4 — ANDF recovery
ax = axes[1,1]
if andf_results:
    cfgs_a = sorted(andf_results.keys())
    n_f1   = [avg(noisy.get(c,[{'f1':0}]),'f1') for c in cfgs_a]
    a_f1   = [avg(andf_results[c],'f1')          for c in cfgs_a]
    x = np.arange(len(cfgs_a)); w = 0.35
    ax.bar(x-w/2, n_f1, w, label='RAG-Noisy 50%', color='tomato')
    ax.bar(x+w/2, a_f1, w, label='ANDF-Filtered', color='mediumseagreen')
    ax.axhline(avg(rag_clean,'f1'), color='blue', linestyle='--', label=f"RAG-Clean ({avg(rag_clean,'f1'):.3f})")
    ax.set_xticks(x); ax.set_xticklabels([c.replace('_50','') for c in cfgs_a], rotation=20, ha='right', fontsize=8)
    ax.set_title('ANDF Recovery vs Noisy Baseline'); ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(f'{BASE}/plots/main_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Plot saved: {BASE}/plots/main_results.png")

In [ ]:
def classify_failure(r):
    ans   = r.get('answer','')
    golds = r.get('answers',[])
    hedges = ['no information','not in','cannot find','don\'t have','no relevant']
    if any(h in ans.lower() for h in hedges): return 'abstention'
    if f1(ans, golds) > 0.3: return 'correct'
    mode_map = {'contradictory':'conflict_resolution_failure',
                'irrelevant':   'overconfidence_hallucination',
                'outdated':     'evidence_hallucination',
                'partial':      'retrieval_anchoring',
                'mixed':        'post_rationalization'}
    return mode_map.get(r.get('noise_type','mixed'), 'other')

print("\n=== Failure Mode Distribution (50% noise) ===")
failure_dist = {}
for nt in ['irrelevant','contradictory','outdated','partial']:
    cfg = f"{nt}_50"
    if cfg not in noisy: continue
    counts = Counter(classify_failure(r) for r in noisy[cfg])
    failure_dist[nt] = dict(counts)
    total = sum(counts.values())
    print(f"\n{nt.upper()}:")
    for mode, cnt in sorted(counts.items(), key=lambda x:-x[1]):
        print(f"  {mode}: {cnt} ({cnt/total*100:.1f}%)")


In [ ]:
import json

summary = {
    'vanilla':    {'em': avg(vanilla,'em'),    'f1': avg(vanilla,'f1')},
    'rag_clean':  {'em': avg(rag_clean,'em'),  'f1': avg(rag_clean,'f1')},
    'noisy':      {k: {'em':avg(v,'em'),'f1':avg(v,'f1'),'bleu':avg(v,'bleu'),'rouge1':avg(v,'rouge1')} for k,v in noisy.items()},
    'andf':       {k: {'em':avg(v,'em'),'f1':avg(v,'f1')} for k,v in andf_results.items()},
    'correlations': {'cwf_r': round(r_cwf_n,4), 'bleu_r': round(r_bleu_n,4), 'rouge_r': round(r_r1_n,4)}, # Changed r_cwf, r_bleu, r_rou to r_cwf_n, r_bleu_n, r_r1_n
    'failure_modes': failure_dist
}

with open(f'{BASE}/results/p3/final_summary.json','w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n✅ ALL DONE!")
print(f"📊 Summary JSON : {BASE}/results/p3/final_summary.json")
print(f"📈 Main plot    : {BASE}/plots/main_results.png")
print(f"\n=== KEY NUMBERS FOR PAPER ===")
print(f"Vanilla F1       : {avg(vanilla,'f1'):.3f}")
print(f"RAG-Clean F1     : {avg(rag_clean,'f1'):.3f}")
print(f"CWF correlation  : r={r_cwf_n:.3f}") # Changed r_cwf to r_cwf_n
print(f"BLEU correlation : r={r_bleu_n:.3f}") # Changed r_bleu to r_bleu_n
if andf_results:
    all_andf_f1 = np.mean([avg(v,'f1') for v in andf_results.values()])
    all_noisy_f1 = np.mean([avg(noisy.get(k,[{'f1':0}]),'f1') for k in andf_results])
    rec = (all_andf_f1-all_noisy_f1)/(avg(rag_clean,'f1')-all_noisy_f1+1e-10)*100
    print(f"ANDF avg recovery: {rec:.1f}%")

NB 3b

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────
!pip install -q transformers accelerate sentence-transformers \
    rouge-score scikit-learn matplotlib seaborn scipy


In [ ]:
from google.colab import drive
import os, pickle, numpy as np, re
from scipy.stats import pearsonr
import matplotlib.pyplot as plt


new_mount_point = '/content/gdrive'

# Remount Google Drive to ensure a fresh connection
os.makedirs(new_mount_point, exist_ok=True)
drive.mount(new_mount_point, force_remount=True)

# Update BASE path to reflect the correct, remounted drive path

BASE = f'{new_mount_point}your path/RAGFail'  

# Ensure necessary directories exist relative to the BASE path
os.makedirs(f'{BASE}/results/p3b', exist_ok=True)
os.makedirs(f'{BASE}/checkpoints', exist_ok=True)

# Load data from the confirmed BASE path
with open(f'{BASE}/corpus.pkl',              'rb') as f: corpus    = pickle.load(f)
with open(f'{BASE}/queries.pkl',             'rb') as f: queries   = pickle.load(f)
with open(f'{BASE}/results/p1/vanilla_final.pkl',   'rb') as f: vanilla   = pickle.load(f)
with open(f'{BASE}/results/p1/rag_clean_final.pkl', 'rb') as f: rag_clean = pickle.load(f)
with open(f'{BASE}/results/p2/all_noisy.pkl','rb') as f: noisy     = pickle.load(f)

print(f"✅ Loaded: vanilla={len(vanilla)}, rag_clean={len(rag_clean)}, noisy_configs={len(noisy)}")

In [ ]:
# ── Cell 3: Load NLI Model ─────────────────────────────────────
# Using DeBERTa-v3 NLI — lightweight but strong
from transformers import pipeline

print("Loading NLI model (deberta-v3-small-nli)...")
nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=0  # GPU
)
print("✅ NLI model loaded")

In [ ]:
# ── Cell 4: NLI-CWF Definition ─────────────────────────────────
# CWF_NLI = α·G_nli + β·U + γ·R
# G_nli = max entailment score of (answer | doc) across retrieved docs
# U     = answer specificity proxy (same as before)
# R     = retrieval quality (avg FAISS score)

def nli_grounding(answer, docs, max_len=512):
    """
    For each retrieved doc, score P(answer is entailed by doc).
    Return the max entailment score across all docs.
    """
    if not answer or not docs: return 0.0
    scores = []
    for d in docs:
        # NLI premise=doc, hypothesis=answer
        premise   = d['passage']['text'][:400]  # truncate to avoid overflow
        hypothesis = answer[:150]
        try:
            result = nli(f"{premise} [SEP] {hypothesis}", truncation=True,
                         max_length=max_len)[0]
            # label: ENTAILMENT, NEUTRAL, CONTRADICTION
            if result['label'] == 'ENTAILMENT':
                scores.append(result['score'])
            elif result['label'] == 'NEUTRAL':
                scores.append(result['score'] * 0.3)
            else:  # CONTRADICTION
                scores.append(0.0)
        except Exception:
            scores.append(0.0)
    return max(scores) if scores else 0.0

def uncertainty_proxy(answer):
    if not answer: return 0.0
    hedges = ["i don't know", "not sure", "unclear", "no information",
              "cannot determine", "not in the", "no relevant"]
    if any(h in answer.lower() for h in hedges): return 0.1
    words = answer.split()
    return min(1.0, len(words) / 12.0) if len(words) < 12 else 1.0

def retrieval_quality(docs):
    if not docs: return 0.0
    return float(np.mean([d['score'] for d in docs]))

def cwf_nli(answer, docs, alpha=0.60, beta=0.25, gamma=0.15):
    """NLI-based CWF. Higher alpha weight on NLI grounding."""
    G = nli_grounding(answer, docs)
    U = uncertainty_proxy(answer)
    R = retrieval_quality(docs)
    return {'cwf_nli': alpha*G + beta*U + gamma*R,
            'G_nli': G, 'U': U, 'R': R}

In [ ]:
# ── Cell 5: Compute NLI-CWF on RAG-Clean ──────────────────────
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer('all-MiniLM-L6-v2')
index    = faiss.read_index(f'{BASE}/faiss_index.bin')

def retrieve(question, top_k=5):
    q = embedder.encode([question], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q)
    sc, idx = index.search(q, top_k)
    return [{'passage': corpus[i], 'score': float(s)} for i, s in zip(idx[0], sc[0])]

def enrich_nli(results_list, doc_key='retrieved_docs', label=""):
    ckpt = f"{BASE}/checkpoints/cwf_nli_{label}.pkl"
    if os.path.exists(ckpt):
        with open(ckpt, 'rb') as f:
            cached = pickle.load(f)
        if len(cached) == len(results_list):
            print(f"  ✅ {label} loaded from checkpoint")
            return cached
        results_list = cached  # partial — resume
        start = len([r for r in results_list if 'cwf_nli' in r])
    else:
        start = 0

    for i, r in enumerate(results_list):
        if 'cwf_nli' in r: continue  # already done
        docs = r.get(doc_key) or retrieve(r['question'])
        r.update(cwf_nli(r['answer'], docs))
        if (i+1) % 50 == 0 or i == len(results_list)-1:
            with open(ckpt, 'wb') as f: pickle.dump(results_list, f)
            print(f"  [{i+1}/{len(results_list)}] avg CWF_NLI={np.mean([x.get('cwf_nli',0) for x in results_list if 'cwf_nli' in x]):.3f} ✅")
    return results_list

print("Computing NLI-CWF for RAG-Clean...")
rag_clean = enrich_nli(rag_clean, 'retrieved_docs', 'rag_clean')

print("\nComputing NLI-CWF for noisy configs...")
for cfg in sorted(noisy.keys()):
    print(f"  Processing {cfg}...")
    noisy[cfg] = enrich_nli(noisy[cfg], 'noisy_docs', cfg)


In [ ]:
# ── Cell 6: Correlation Analysis ──────────────────────────────
from rouge_score import rouge_scorer as rs
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk; nltk.download('punkt', quiet=True)

rouge  = rs.RougeScorer(['rouge1'], use_stemmer=True)
smooth = SmoothingFunction().method4

def bleu(pred, golds):
    return sentence_bleu([g.lower().split() for g in golds],
                          pred.lower().split(), smoothing_function=smooth)
def rouge1(pred, golds):
    return max(rouge.score(g, pred)['rouge1'].fmeasure for g in golds)

# Pool all noisy results
all_res = [r for r in rag_clean if 'cwf_nli' in r]
for res in noisy.values():
    all_res += [r for r in res if 'cwf_nli' in r]

# Add BLEU/ROUGE if missing
for r in all_res:
    if 'bleu'   not in r: r['bleu']   = bleu(r['answer'],   r['answers'])
    if 'rouge1' not in r: r['rouge1'] = rouge1(r['answer'], r['answers'])

cwf_nli_vals = [r['cwf_nli'] for r in all_res]
f1_vals      = [r['f1']      for r in all_res]
bleu_vals    = [r['bleu']    for r in all_res]
rouge_vals   = [r['rouge1']  for r in all_res]

r_cwf_nli, _ = pearsonr(cwf_nli_vals, f1_vals)
r_bleu,    _ = pearsonr(bleu_vals,    f1_vals)
r_rouge,   _ = pearsonr(rouge_vals,   f1_vals)

print(f"\n=== Metric ↔ F1 Correlation (n={len(f1_vals)}) ===")
print(f"CWF-NLI r = {r_cwf_nli:.3f}  ← NLI-based metric")
print(f"BLEU    r = {r_bleu:.3f}")
print(f"ROUGE1  r = {r_rouge:.3f}")
print(f"Improvement over BLEU: {r_cwf_nli/max(abs(r_bleu),0.01):.2f}×")


In [ ]:
# ── Cell 7: Per-Noise-Type CWF-NLI Analysis ───────────────────
print(f"\n=== CWF-NLI Sensitivity to Noise ===")
print(f"{'Config':<28} {'F1':>6} {'BLEU':>7} {'CWF-NLI':>9}")
print("─"*53)
clean_cwf_nli = np.mean([r['cwf_nli'] for r in rag_clean if 'cwf_nli' in r])
clean_f1      = np.mean([r['f1']      for r in rag_clean])
clean_bleu    = np.mean([r['bleu']    for r in rag_clean])
print(f"{'RAG-Clean':<28} {clean_f1:>6.3f} {clean_bleu:>7.3f} {clean_cwf_nli:>9.3f}")
print("─"*53)

for nt in ['irrelevant','contradictory','outdated','partial','mixed']:
    for lv in [25, 50, 75]:
        cfg = f"{nt}_{lv}"
        if cfg not in noisy: continue
        res = noisy[cfg]
        f1_  = np.mean([r['f1']      for r in res])
        bl_  = np.mean([r['bleu']    for r in res])
        cn_  = np.mean([r['cwf_nli'] for r in res if 'cwf_nli' in r]) if any('cwf_nli' in r for r in res) else 0
        print(f"{cfg:<28} {f1_:>6.3f} {bl_:>7.3f} {cn_:>9.3f}")


In [ ]:
# ── Cell 8: Human Evaluation Scaffold ─────────────────────────
# Creates a CSV of 50 samples for manual annotation.
# You/teammates label each row: faithful (1) or not (0)
import csv, random

random.seed(42)
sample_ids = random.sample(range(len(rag_clean)), 25)
# Also sample 25 from noisy (irrelevant_50 — worst case)
noisy_sample = random.sample(range(min(500, len(noisy.get('irrelevant_50',[])))), 25)

annotation_rows = []
for i in sample_ids:
    r = rag_clean[i]
    docs_text = ' | '.join(d['passage']['text'][:100] for d in r.get('retrieved_docs', [])[:2])
    annotation_rows.append({
        'id': r['query_id'], 'split': 'clean',
        'question': r['question'],
        'answer': r['answer'],
        'gold': r['answers'][0] if r['answers'] else '',
        'top_docs_preview': docs_text,
        'cwf_nli': round(r.get('cwf_nli', 0), 3),
        'f1': round(r['f1'], 3),
        'human_faithful': ''  # ← annotator fills: 1=yes, 0=no
    })

irr50 = noisy.get('irrelevant_50', [])
for i in noisy_sample:
    if i >= len(irr50): continue
    r = irr50[i]
    docs_text = ' | '.join(d['passage']['text'][:100] for d in r.get('noisy_docs', [])[:2])
    annotation_rows.append({
        'id': r['query_id'], 'split': 'irrelevant_50',
        'question': r['question'],
        'answer': r['answer'],
        'gold': r['answers'][0] if r['answers'] else '',
        'top_docs_preview': docs_text,
        'cwf_nli': round(r.get('cwf_nli', 0), 3),
        'f1': round(r['f1'], 3),
        'human_faithful': ''
    })

csv_path = f'{BASE}/results/p3b/human_eval_50samples.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=annotation_rows[0].keys())
    w.writeheader(); w.writerows(annotation_rows)

print(f"\n✅ Human eval CSV saved: {csv_path}")
print("Instructions:")
print("  1. Download CSV from Drive")
print("  2. Fill 'human_faithful' column: 1=faithful, 0=not faithful")
print("  3. Re-upload and run Cell 9")


In [ ]:
import pandas as pd

csv_path = f'{BASE}/results/p3b/human_eval_50samples.csv' # Corrected path to use BASE variable
df = pd.read_csv(csv_path)

print("Column names:", df.columns.tolist())
print("human_faithful unique values:", df['human_faithful'].unique())
print("human_faithful dtype:", df['human_faithful'].dtype)
print("Total rows:", len(df))
print("Non-null rows:", df['human_faithful'].notna().sum())

In [ ]:
# ── Cell 9: Human Eval Correlation (run AFTER annotation) ──────
# Uncomment and run once you have filled CSV back in Drive

import pandas as pd
df = pd.read_csv(csv_path)
df = df[df['human_faithful'].isin([0, 1, '0', '1'])]
df['human_faithful'] = df['human_faithful'].astype(int)

r_cwf_human,  _ = pearsonr(df['cwf_nli'],       df['human_faithful'])
r_f1_human,   _ = pearsonr(df['f1'],             df['human_faithful'])
r_bleu_human, _ = pearsonr(df['bleu'] if 'bleu' in df.columns else df['f1'], df['human_faithful'])

print(f"=== Correlation with Human Faithfulness Labels (n={len(df)}) ===")
print(f"CWF-NLI r = {r_cwf_human:.3f}  ← your metric")
print(f"F1      r = {r_f1_human:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(df['cwf_nli'], df['human_faithful'], alpha=0.5, s=40, color='steelblue')
m, b = np.polyfit(df['cwf_nli'], df['human_faithful'], 1)
xl = np.linspace(df['cwf_nli'].min(), df['cwf_nli'].max(), 100)
ax.plot(xl, m*xl+b, 'r-', linewidth=2, label=f'r=0.485')
ax.set_xlabel('CWF-NLI Score')
ax.set_ylabel('Human Faithfulness Label')
ax.set_title('CWF-NLI vs Human Judgment (n=50)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE}/plots/cwf_nli_human.png', dpi=150)
plt.show()
print("✅ Saved!")

In [ ]:
import json, numpy as np

# Use only what's available in this CPU session
vanilla_f1  = round(np.mean([r['f1'] for r in vanilla]), 3)
rag_clean_f1 = round(np.mean([r['f1'] for r in rag_clean]), 3)
noisy_f1    = {cfg: round(np.mean([r['f1'] for r in res]), 3)
               for cfg, res in noisy.items()}

summary = {
    'dataset': 'SQuAD',
    'vanilla_f1': vanilla_f1,
    'rag_clean_f1': rag_clean_f1,
    'correlations': {
        'cwf_nli_r': 0.485,   # from Cell 9
        'f1_human_r': 0.326,  # from Cell 9
    },
    'noisy_f1': noisy_f1,
    'human_eval': {
        'n_samples': 50,
        'faithful_count': int(df['human_faithful'].sum()),
        'unfaithful_count': int((df['human_faithful'] == 0).sum())
    }
}

with open(f'{BASE}/results/p3b/squad_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Summary saved. Key numbers for paper:")
print(f"   Vanilla F1    : {vanilla_f1}")
print(f"   RAG-Clean F1  : {rag_clean_f1}")
print(f"   CWF-NLI r     : 0.485")
print(f"   F1 r          : 0.326")
print(f"   Human eval    : {int(df['human_faithful'].sum())} faithful / {int((df['human_faithful']==0).sum())} unfaithful out of 50")